# Week 41, Day 4: Sliding window

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w041d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
text = "abcabcbb"
seen = {}
start = best = 0
for end, ch in enumerate(text):
    if ch in seen and seen[ch] >= start:
        start = seen[ch] + 1
    seen[ch] = end
    best = max(best, end - start + 1)
print(best, text[start:end + 1])

In [ ]:
def max_average(nums, k):
    window = sum(nums[:k])
    best = window
    for i in range(k, len(nums)):
        window += nums[i] - nums[i - k]        # in comes nums[i], out goes nums[i - k]
        best = max(best, window)
    return best / k

print(max_average([1, 12, -5, -6, 50, 3], 4))

In [ ]:
from collections import Counter

def anagram_starts(text, word):
    k = len(word)
    want, window = Counter(word), Counter(text[:k])
    starts = [0] if window == want else []
    for i in range(k, len(text)):
        window[text[i]] += 1                    # in
        window[text[i - k]] -= 1                # out
        if window[text[i - k]] == 0:
            del window[text[i - k]]             # keep the comparison exact
        if window == want:
            starts.append(i - k + 1)
    return starts

print(anagram_starts("cbaebabacd", "abc"))

In [ ]:
def longest_k_distinct(text, k):
    counts = {}
    start = 0
    best = ""
    for end, ch in enumerate(text):
        counts[ch] = counts.get(ch, 0) + 1
        while len(counts) > k:
            out = text[start]
            counts[out] -= 1
            if counts[out] == 0:
                del counts[out]
            start += 1
        if end - start + 1 > len(best):
            best = text[start:end + 1]
    return best

print(longest_k_distinct("eceba", 2), longest_k_distinct("aabbcc", 1), longest_k_distinct("aabacbebebe", 3))

In [ ]:
def shortest_at_least(nums, target):
    start, total = 0, 0
    best = None
    for end, x in enumerate(nums):
        total += x
        while total >= target:                       # still valid: record it, then try smaller
            if best is None or end - start + 1 < best:
                best = end - start + 1
            total -= nums[start]
            start += 1
    return best

print(shortest_at_least([2, 3, 1, 2, 4, 3], 7), shortest_at_least([1, 1, 1], 5))

In [ ]:
def shortest_at_least(nums, target):
    start, total, best = 0, 0, None
    for end, x in enumerate(nums):
        total += x
        while total >= target:
            if best is None or end - start + 1 < best:
                best = end - start + 1
            total -= nums[start]
            start += 1
    return best

def brute(nums, target):
    lengths = [j - i + 1 for i in range(len(nums)) for j in range(i, len(nums)) if sum(nums[i:j + 1]) >= target]
    return min(lengths, default=None)

nums = [84, -37, 32, 40, 95]
print("window:", shortest_at_least(nums, 167), " every run:", brute(nums, 167))

> **Key idea.** A sliding window keeps two pointers moving the same way and a summary of what's between them. Fixed windows add the item coming in and remove the item going out. Variable windows grow on the right and shrink from the left while invalid (for the longest) or while still valid (for the shortest), recording the best. They're O(n) because `start` only moves forward, at most n times in all. Keep the window's state exact as items enter and leave. Windows need monotonic conditions: with negative numbers, use prefix sums instead.

### Exercise: The busiest hour

`counts` holds the number of visitors in each minute. Write `busiest(counts, k)`: the start minute and total of the busiest run of `k` consecutive minutes, as `(start, total)`, choosing the **earliest** on a tie. If `k` is less than 1 or more than `len(counts)`, raise `ValueError`. Run in O(n): the checks use 200,000 minutes.

In [ ]:
def busiest(counts, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_busiest():\n    \"small\"\n    assert busiest([1, 3, 2, 6, 0, 4], 2) == (2, 8) and busiest([5, 1, 5], 1) == (0, 5) and busiest([4, 4], 2) == (0, 8)\n    assert busiest([1, 2, 3, 1, 2, 3], 3) == (0, 6)\n    for bad in [0, 7]:\n        try:\n            busiest([1, 2, 3, 4, 5, 6], bad)\n            assert False\n        except ValueError:\n            pass\ndef test_big():\n    \"200,000 minutes\"\n    rng = random.Random(5)\n    c = [rng.randint(0, 50) for _ in range(200_000)]\n    start, total = busiest(c, 60)\n    assert total == sum(c[start:start + 60])")

### Exercise: Hidden anagrams

Write `anagram_starts(text, word)`: the start positions of every run of `len(word)` letters in `text` that is an anagram of `word`, in order. Keep a dictionary of the window's letter counts and update it as the window slides (don't re-count each window). A word longer than the text gives `[]`.

In [ ]:
def anagram_starts(text, word):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_small():\n    \"anagrams\"\n    assert anagram_starts(\"cbaebabacd\", \"abc\") == [0, 6] and anagram_starts(\"abab\", \"ab\") == [0, 1, 2]\n    assert anagram_starts(\"ab\", \"abc\") == [] and anagram_starts(\"aaaa\", \"aa\") == [0, 1, 2] and anagram_starts(\"xyz\", \"q\") == []\ndef test_big():\n    \"long text\"\n    rng = random.Random(1)\n    text = \"\".join(rng.choice(\"abc\") for _ in range(100_000))\n    got = anagram_starts(text, \"abcab\")\n    assert got[:3] == [i for i in range(len(text) - 4) if sorted(text[i:i + 5]) == sorted(\"abcab\")][:3] and len(got) > 1000")

### Exercise: The longest run without repeats

Write `longest_unique(text)`: the longest run of characters with no character repeated, returning the first such run if there are several of the same length. Use a variable window with a dictionary of each character's last position, in O(n).

In [ ]:
def longest_unique(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_unique():\n    \"runs\"\n    assert longest_unique(\"abcabcbb\") == \"abc\" and longest_unique(\"bbbbb\") == \"b\" and longest_unique(\"pwwkew\") == \"wke\"\n    assert longest_unique(\"\") == \"\" and longest_unique(\"abba\") == \"ab\" and longest_unique(\"dvdf\") == \"vdf\" and longest_unique(\"tmmzuxt\") == \"mzuxt\"")

### Exercise: The shortest run that reaches a target

Write `shortest_reaching(nums, target)` for a list of **positive** numbers: the length of the shortest run of consecutive numbers whose total is at least `target`, or `0` if there's none. Shrink the window while it's still valid. O(n): the checks use 100,000 numbers.

In [ ]:
def shortest_reaching(nums, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(nums, target):\n    best = 0\n    for i in range(len(nums)):\n        t = 0\n        for j in range(i, len(nums)):\n            t += nums[j]\n            if t >= target:\n                best = j - i + 1 if best == 0 else min(best, j - i + 1)\n                break\n    return best\ndef test_small():\n    \"small\"\n    assert shortest_reaching([2, 3, 1, 2, 4, 3], 7) == 2 and shortest_reaching([1, 4, 4], 4) == 1 and shortest_reaching([1, 1, 1], 5) == 0\n    rng = random.Random(9)\n    for _ in range(200):\n        nums = [rng.randint(1, 9) for _ in range(rng.randint(0, 15))]\n        t = rng.randint(1, 40)\n        assert shortest_reaching(nums, t) == brute(nums, t), (nums, t)\ndef test_big():\n    \"100,000 numbers\"\n    rng = random.Random(3)\n    nums = [rng.randint(1, 100) for _ in range(100_000)]\n    assert shortest_reaching(nums, 5000) > 0")

### Exercise: Flip k zeros

`bits` is a list of 0s and 1s. Write `longest_after_flips(bits, k)`: the length of the longest run of 1s you can make by changing **at most** `k` zeros to ones. The window is valid while it holds at most `k` zeros.

In [ ]:
def longest_after_flips(bits, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(bits, k):\n    return max((j - i for i in range(len(bits) + 1) for j in range(i, len(bits) + 1) if bits[i:j].count(0) <= k), default=0)\ndef test_flips():\n    \"flips\"\n    assert longest_after_flips([1, 1, 0, 0, 1, 1, 1, 0, 1], 1) == 5 and longest_after_flips([0, 0, 0], 0) == 0 and longest_after_flips([0, 0, 0], 5) == 3\n    rng = random.Random(4)\n    for _ in range(200):\n        bits = [rng.randint(0, 1) for _ in range(rng.randint(0, 14))]\n        k = rng.randint(0, 3)\n        assert longest_after_flips(bits, k) == brute(bits, k), (bits, k)")

### Bug hunt: A window that only grows

This fixed window adds each new item but never takes the old one out, so its "window" total is really a running total of everything so far: it prints `14`. Fix it so it prints the best total of 2 neighbours, `9`.

In [ ]:
def best_pair_total(nums):
    window = nums[0] + nums[1]
    best = window
    for i in range(2, len(nums)):
        window += nums[i]
        best = max(best, window)
    return best

print(best_pair_total([1, 4, 5, 2, 2]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"take the old item out\"\n    expect_output(\"9\")")

### Bug hunt: Shrinking once is not enough

To find the **shortest** run reaching the target, the window must keep shrinking while it's still valid. This one shrinks only once per step (`if` instead of `while`), so it never tries the short windows: it prints `5` for target 10 in `[1, 1, 1, 1, 10]`. Fix it so it prints `1`.

In [ ]:
def shortest_reaching(nums, target):
    start, total, best = 0, 0, 0
    for end, x in enumerate(nums):
        total += x
        if total >= target:
            length = end - start + 1
            best = length if best == 0 else min(best, length)
            total -= nums[start]
            start += 1
    return best

print(shortest_reaching([1, 1, 1, 1, 10], 10))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"shrink while valid\"\n    expect_output(\"1\")")

### Bug hunt: Counts that never vanish

This anagram finder lowers a letter's count when it leaves the window but leaves a `0` behind, so `{"a": 1, "b": 1, "c": 0}` never equals `{"a": 1, "b": 1}` and it misses the second match: it prints `[0]`. Delete counts that reach 0, so it prints `[0, 3]`.

In [ ]:
def anagram_starts(text, word):
    k = len(word)
    want, window = {}, {}
    for ch in word:
        want[ch] = want.get(ch, 0) + 1
    for ch in text[:k]:
        window[ch] = window.get(ch, 0) + 1
    starts = [0] if window == want else []
    for i in range(k, len(text)):
        window[text[i]] = window.get(text[i], 0) + 1
        window[text[i - k]] -= 1
        if window == want:
            starts.append(i - k + 1)
    return starts

print(anagram_starts("abcab", "ab"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"exact counts\"\n    expect_output(\"[0, 3]\")")

### Puzzle: The variable window

Arrange the lines into a function giving the length of the longest run of 1s after flipping at most one 0. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Answer without running code.

p = ___   # longest run with at most 2 different letters in "aabbbcbbaa"
q = ___   # how many times start can move, in total, in a variable window over n items: "at most n" or "about n^2"
r = ___   # the best total of 3 neighbours in [4, -1, 2, 1, -5, 4]
s = ___   # True or False: shrink-while-valid finds the shortest run with sum >= target when numbers can be negative
t = ___   # the length of the longest run without repeats in "abcdeafgh"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 6 and q == \"at most n\" and r == 5, \"p, q and r\"\n    assert s is False and t == 8, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the smallest window with every letter

Write `smallest_cover(text, need)`: the shortest run of `text` that contains every character of `need`, **with repeats** (if `need` is `"aab"`, the run needs two `a`s and a `b`). Return the run (the first of the shortest if there are several), or `""` if there's none, or if `need` is empty. Use a window with two dictionaries of counts and a running total of how many characters of `need` are covered, so each step is O(1); the checks use a text of 100,000 characters.

In [ ]:
def smallest_cover(text, need):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(text, need):\n    from collections import Counter\n    want = Counter(need)\n    best = \"\"\n    for i in range(len(text)):\n        for j in range(i + 1, len(text) + 1):\n            if not want - Counter(text[i:j]):\n                if not best or j - i < len(best):\n                    best = text[i:j]\n                break\n    return best if need else \"\"\ndef test_small():\n    \"small\"\n    assert smallest_cover(\"ADOBECODEBANC\", \"ABC\") == \"BANC\" and smallest_cover(\"a\", \"aa\") == \"\" and smallest_cover(\"aa\", \"aa\") == \"aa\"\n    assert smallest_cover(\"abc\", \"\") == \"\" and smallest_cover(\"xyzzy\", \"zy\") == \"yz\"\n    rng = random.Random(6)\n    for _ in range(150):\n        text = \"\".join(rng.choice(\"abcd\") for _ in range(rng.randint(0, 14)))\n        need = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(1, 3)))\n        assert smallest_cover(text, need) == brute(text, need), (text, need)\ndef test_big():\n    \"100,000 characters\"\n    rng = random.Random(2)\n    text = \"\".join(rng.choice(\"abcdefghij\") for _ in range(100_000))\n    got = smallest_cover(text, \"jihgfedcba\")\n    assert len(got) >= 10 and set(got) >= set(\"abcdefghij\")")

## Warm-up

The longest run of different characters is 3 long. The window `[start, end]` only ever moves to the right: `end` steps through every character, and `start` jumps forward past a repeat. By the end, the window has moved on to the last valid run, just `"b"`, while `best` remembers the length 3.

## Fixed windows

A window of a fixed size `k` slides one step at a time. Instead of adding up all `k` items again at each position (O(nk)), **add the item coming in and subtract the item going out** (O(n)):

The same idea works for anything you can **update** as items come in and go out: a total, a count, a dictionary of counts. For example, a word's anagrams in a text are the windows whose letter counts match the word's:

## Windows that stretch and shrink

Many problems ask for the **longest** or **shortest** run that meets a condition. A variable window answers them with one pattern:

```python
start = 0
for end in range(len(items)):              # 1. grow: take items[end] into the window
    add items[end] to the window
    while the window is not valid:         # 2. shrink from the left until it is
        remove items[start] from the window
        start += 1
    record the window [start, end]         # 3. it's valid here: is it the best?
```

For "the longest run with at most `k` different values":

For the **shortest** run, shrink as long as the window is still valid, recording it each time. "The shortest run of positive numbers adding up to at least a target":

## Why it's O(n)

A loop inside a loop usually means O(n²). Here it doesn't, because of what the inner loop does: **every time it runs, `start` moves forward by one, and `start` never moves back.** Over the whole run, `start` moves at most n times in total, and `end` moves exactly n times. So the total work is at most 2n steps: O(n), whatever the data. This is amortised reasoning (Day 2): one outer step may shrink a lot, but the shrinking is shared out across the whole loop.

The same argument fails if `start` could move backwards, or be reset to `end` and walk forward again; check for that whenever you write a window.

## Counting inside a window

The window's **state** is whatever you need to decide whether it's valid, kept up to date as items enter and leave:

| Question | State | Valid when |
|---|---|---|
| longest run with no repeats | last position of each character, or a set | no repeats inside |
| at most k different values | counts of each value (dict) | `len(counts) <= k` |
| at most k zeros (flip k zeros to 1s) | number of zeros inside | `zeros <= k` |
| shortest run with sum ≥ target | running total | `total >= target` |
| a window containing every letter of a word | counts, and how many letters are fully covered | covered letters == letters needed |

Keeping the state exact (deleting counts that reach 0, for example) is where most window bugs live.

## When a window fails

The shrink-from-the-left pattern relies on **monotonicity**: adding an item can only make a window "more" (a bigger total, more distinct values), and removing one can only make it "less". With negative numbers that breaks: removing a negative number makes the total **bigger**, so "shrink while total ≥ target" can stop too soon or too late.

The window answers 5; the true answer is 3 (`[32, 40, 95]`). When values can be negative, tomorrow's **prefix sums** (with a dictionary or a sorted structure) are the right tool.

## Practice

## Quiz

## Daily challenge

## Before you go

Write the variable-window skeleton from memory, then decide, for each problem in the state table above, what "valid" means and whether you shrink while invalid or while valid. Tomorrow: prefix sums, for range questions, and for the window problems that windows can't solve.